### How To: Hold Sequences Packed

A one-hot encoding spends four values on every position of a sequence: four bytes in int8, sixteen in float32. That is fine for a batch, but it adds up quickly when you hold many sequences at once. The 2,114bp windows around 100,000 peaks take 846 MB as int8, and twenty dinucleotide shuffles of each take 17 GB. A sequence of A, C, G and T only needs two bits a position, so tangermeme can hold sequences as a `PackedSequence`, which stores four positions in each byte, sixteen times less than int8.

By the end of this how-to you will be able to make packed sequences directly from a FASTA file or a string, pass them to any tangermeme function that takes one-hot sequences, get packed sequences back from the functions that make sequences, and save and load them. Each of those functions gives the same result for a packed sequence as for its one-hot encoding, so you can switch without changing anything else.

#### The smallest example

`PackedSequence.from_one_hot` packs a one-hot encoding of shape `(n, 4, length)`, and `to_one_hot` decodes it.

In [1]:
import torch

from tangermeme.utils import random_one_hot
from tangermeme.utils import PackedSequence

X = random_one_hot((1000, 4, 2114), random_state=0)
X_packed = PackedSequence.from_one_hot(X)

print(X_packed)
print("int8 one-hot: {:,} bytes, packed: {:,} bytes".format(X.nbytes, X_packed.data.nbytes))
print("decodes to the same encoding:", torch.equal(X_packed.to_one_hot(), X))

PackedSequence(shape=(1000, 4, 2114), n_runs=0, alphabet=ACGT)
int8 one-hot: 8,456,000 bytes, packed: 529,000 bytes
decodes to the same encoding: True


The packed bytes are in `X_packed.data`, a `torch.uint8` tensor of shape `(1000, 529)`: 2,114 positions take 529 bytes, the last of which holds two positions and two bits of padding. `X_packed.shape` gives the shape of the one-hot encoding it stands for, so code that reads `X.shape` works on either.

#### Making packed sequences directly

Packing a one-hot encoding means building that encoding first. The functions that make sequences can skip it with `return_packed=True`: `one_hot_encode` for a string, `random_one_hot` for random sequences, and `extract_loci` for windows of a FASTA file, which it reads and packs a chunk of windows at a time so the one-hot encoding of all of them is never held. Here we write a small FASTA file with two chromosomes to read windows from.

In [2]:
import numpy
import pandas
import torch

from tangermeme.io import extract_loci
from tangermeme.utils import one_hot_encode

rng = numpy.random.RandomState(0)
with open("example.fa", "w") as outfile:
	for name in ("chr1", "chr2"):
		sequence = ''.join(rng.choice(list("ACGT"), size=5000))
		outfile.write(">{}\n".format(name))
		for start in range(0, len(sequence), 60):
			outfile.write(sequence[start:start+60] + "\n")

loci = pandas.DataFrame({"chrom": ["chr1", "chr1", "chr2"], "start": [1000, 2500, 3000],
	"end": [1100, 2600, 3100]})

X = extract_loci(loci, "example.fa", in_window=500)
X_packed = extract_loci(loci, "example.fa", in_window=500, return_packed=True)
print(X_packed)
print("same windows:", torch.equal(X_packed.to_one_hot(), X))

x_packed = one_hot_encode("ACGTTGCA", return_packed=True)
print(x_packed, x_packed.data)

PackedSequence(shape=(3, 4, 500), n_runs=0, alphabet=ACGT)
same windows: True
PackedSequence(shape=(4, 8), n_runs=0, alphabet=ACGT) tensor([ 27, 228], dtype=torch.uint8)


The single sequence from `one_hot_encode` has no batch axis, just as `one_hot_encode` returns a tensor of shape `(4, length)`. Its two bytes are `0b00011011 = 27` for ACGT and `0b11100100 = 228` for TGCA: each character is stored as its index in the alphabet, the first of a byte's four positions in its two most significant bits.

#### Unknown characters

Two bits leave no code for an unknown character such as N, which a one-hot encoding stores as an all-zero column. A `PackedSequence` keeps the runs of them in `n_runs`, one row of `(sequence, start, end)` per run, and decodes them back to all-zero columns.

In [3]:
import torch

from tangermeme.utils import one_hot_encode
from tangermeme.utils import PackedSequence

X = torch.stack([one_hot_encode("ACGTNNNNACGT"), one_hot_encode("NACGTACGTACN")])
X_packed = PackedSequence.from_one_hot(X)

print(X_packed.n_runs)
print("decodes to the same encoding:", torch.equal(X_packed.to_one_hot(), X))

tensor([[ 0,  4,  8],
        [ 1,  0,  1],
        [ 1, 11, 12]])
decodes to the same encoding: True


Assembly gaps are usually rare inside peaks, so `n_runs` stays small. A sequence that is mostly N still packs, as one run.

#### Indexing and decoding

A `PackedSequence` is indexed along its batch axes the way a tensor of those axes would be, and indexing returns a `PackedSequence`. That is how you decode only what you need: select a batch, then call `to_one_hot`, which takes the `dtype` and the `device` to decode on. On a GPU, only the packed bytes are moved before decoding.

In [4]:
import torch

from tangermeme.utils import random_one_hot
from tangermeme.utils import PackedSequence

X = random_one_hot((1000, 4, 2114), random_state=0)
X_packed = PackedSequence.from_one_hot(X)

X_batch = X_packed[100:164].to_one_hot(dtype=torch.float32)
print(X_batch.shape, X_batch.dtype)
print("same batch:", torch.equal(X_batch, X[100:164].float()))

print(X_packed[[5, 5, 900]])
print(X_packed.reshape(10, 100))

torch.Size([64, 4, 2114]) torch.float32
same batch: True
PackedSequence(shape=(3, 4, 2114), n_runs=0, alphabet=ACGT)
PackedSequence(shape=(10, 100, 4, 2114), n_runs=0, alphabet=ACGT)


`to_codes` gives the index of each character instead, as `torch.uint8` with 4 for an unknown character, which is often all an analysis needs.

#### Passing packed sequences to tangermeme functions

Every function that takes one-hot sequences as `X` takes a `PackedSequence`, and gives the result of its one-hot encoding. `predict` and `deep_lift_shap` decode one batch at a time, so the one-hot encoding of all the sequences is never held. To show the results are the same, we define a small convolutional model.

In [5]:
import torch

from tangermeme.utils import random_one_hot
from tangermeme.utils import PackedSequence
from tangermeme.predict import predict
from tangermeme.deep_lift_shap import deep_lift_shap

torch.manual_seed(0)
model = torch.nn.Sequential(torch.nn.Conv1d(4, 16, 21, padding=10), torch.nn.ReLU(),
	torch.nn.Conv1d(16, 16, 11, padding=5), torch.nn.ReLU(), torch.nn.AdaptiveAvgPool1d(1),
	torch.nn.Flatten(), torch.nn.Linear(16, 1))

X = random_one_hot((64, 4, 1000), random_state=0).float()
X_packed = PackedSequence.from_one_hot(X)

y = predict(model, X, device='cpu')
y_packed = predict(model, X_packed, device='cpu')
print("same predictions:", torch.equal(y_packed, y))

X_attr = deep_lift_shap(model, X[:8], n_shuffles=10, random_state=0, device='cpu')
X_attr_packed = deep_lift_shap(model, X_packed[:8], n_shuffles=10, random_state=0, device='cpu')
print("same attributions:", torch.equal(X_attr_packed, X_attr))

same predictions: True


same attributions: True


The default references of `deep_lift_shap` are drawn from each decoded batch with the same seeds, so the attributions match to the bit. Precomputed references can be packed too, which is where most of the memory goes: twenty shuffles of every sequence. `dinucleotide_shuffle` packs each shuffle as it draws it, so the shuffles are never held one-hot.

In [6]:
import torch

from tangermeme.utils import random_one_hot
from tangermeme.utils import PackedSequence
from tangermeme.ersatz import dinucleotide_shuffle
from tangermeme.deep_lift_shap import deep_lift_shap

torch.manual_seed(0)
model = torch.nn.Sequential(torch.nn.Conv1d(4, 16, 21, padding=10), torch.nn.ReLU(),
	torch.nn.Conv1d(16, 16, 11, padding=5), torch.nn.ReLU(), torch.nn.AdaptiveAvgPool1d(1),
	torch.nn.Flatten(), torch.nn.Linear(16, 1))
X = random_one_hot((8, 4, 1000), random_state=0).float()

references = dinucleotide_shuffle(X, n=20, random_state=0)
references_packed = dinucleotide_shuffle(X, n=20, random_state=0, return_packed=True)
print(references_packed)
print("references: {:,} bytes as float32, {:,} packed".format(references.nbytes,
	references_packed.data.nbytes))

X_attr = deep_lift_shap(model, X, references=references, device='cpu')
X_attr_packed = deep_lift_shap(model, PackedSequence.from_one_hot(X), references=references_packed,
	device='cpu')
print("same attributions:", torch.equal(X_attr_packed, X_attr))

PackedSequence(shape=(8, 20, 4, 1000), n_runs=0, alphabet=ACGT)
references: 2,560,000 bytes as float32, 40,000 packed
same attributions: True


A `PackedSequence` with several batch axes, here `(8, 20)` for eight sequences and their twenty shuffles, is indexed as `references_packed[i, j]`, which is how `deep_lift_shap` reads one batch of example-reference pairs at a time. Set `n_jobs` on `dinucleotide_shuffle`, for example to 4, to split the sequences across threads; the shuffles are the same for any number of threads.

#### Packed sequences in, packed sequences out

The functions that make new sequences from `X`, those in `ersatz`, return a `PackedSequence` when given one, and return one for a one-hot `X` with `return_packed=True`. `marginalize`, `ablate` and `space` use them, so a packed `X` keeps the copy of the sequences with a motif or a shuffled region packed as well, and pass it on to `func`.

In [7]:
import torch

from tangermeme.utils import random_one_hot
from tangermeme.utils import PackedSequence
from tangermeme.ersatz import substitute
from tangermeme.marginalize import marginalize
from tangermeme.predict import predict

torch.manual_seed(0)
model = torch.nn.Sequential(torch.nn.Conv1d(4, 16, 21, padding=10), torch.nn.ReLU(),
	torch.nn.Conv1d(16, 16, 11, padding=5), torch.nn.ReLU(), torch.nn.AdaptiveAvgPool1d(1),
	torch.nn.Flatten(), torch.nn.Linear(16, 1))
X = random_one_hot((64, 4, 1000), random_state=0).float()
X_packed = PackedSequence.from_one_hot(X)

X_motif = substitute(X_packed, "GATAAG", start=500)
print(X_motif)

y_before, y_after = marginalize(model, X, "GATAAG", device='cpu')
y_before_packed, y_after_packed = marginalize(model, X_packed, "GATAAG", device='cpu')
print("same marginalization:", torch.equal(y_after_packed, y_after))

def my_func(model, X, **kwargs):
	print("my_func received", type(X).__name__, X.dtype)
	return predict(model, X, **kwargs)

_ = marginalize(model, X_packed, "GATAAG", func=my_func, device='cpu')

PackedSequence(shape=(64, 4, 1000), n_runs=0, alphabet=ACGT)
same marginalization: True
my_func received Tensor torch.int8
my_func received Tensor torch.int8


Your own `func`, or `shuffle_fn` in `ablate`, receives the int8 one-hot encoding rather than the `PackedSequence`, so it does not need to know about packing. Only tangermeme's own functions that take a `PackedSequence`, such as `predict`, `deep_lift_shap`, `pisa`, `saturation_mutagenesis` and the shuffles in `ersatz`, are given it packed, including when wrapped in `functools.partial`. Predictions and attributions are returned as tensors either way: only sequences come back packed.

#### Saving and loading

`save` writes a directory of `.npy` files and `load` reads it back. With `mmap=True` the bytes are memory-mapped rather than read, so indexing reads only the sequences it selects from disk.

In [8]:
import os
import torch

from tangermeme.utils import random_one_hot
from tangermeme.utils import PackedSequence

X = random_one_hot((1000, 4, 2114), random_state=0)
PackedSequence.from_one_hot(X).save("example_packed")
print(sorted(os.listdir("example_packed")))

X_packed = PackedSequence.load("example_packed", mmap=True)
print("same batch:", torch.equal(X_packed[200:264].to_one_hot(), X[200:264]))

['data.npy', 'meta.json', 'n_runs.npy']
same batch: True


#### What cannot be packed

A `PackedSequence` holds exactly one character, or an unknown one, at each position of an alphabet of four characters. Soft encodings, such as PWMs or averaged sequences, cannot be packed, and neither can alphabets of other sizes, such as protein. Both raise a `ValueError`, from `from_one_hot` or from `return_packed=True`, rather than lose information.

In [9]:
import torch

from tangermeme.utils import random_one_hot
from tangermeme.utils import PackedSequence

X_soft = torch.full((2, 4, 100), 0.25)
try:
	PackedSequence.from_one_hot(X_soft)
except ValueError as e:
	print("ValueError:", e)

try:
	random_one_hot((2, 20, 100), return_packed=True)
except ValueError as e:
	print("ValueError:", e)

ValueError: X must be one-hot encoded.
ValueError: return_packed=True needs an alphabet of four characters, but shape has 20.


#### Summary

- A `PackedSequence` holds sequences of a four-character alphabet at two bits a position, sixteen times less than an int8 one-hot encoding, with unknown characters kept as runs.
- `return_packed=True` on `extract_loci`, `one_hot_encode`, `random_one_hot` and the functions in `ersatz` makes packed sequences without holding their one-hot encoding.
- Every function that takes one-hot sequences as `X` takes a `PackedSequence` and gives the result of its one-hot encoding; functions that take PWMs, attributions or predictions, such as `entropy` or `unchunk`, do not. Sequences come back packed; predictions and attributions are tensors as before.
- Indexing selects along the batch axes and returns a `PackedSequence`, and `to_one_hot(dtype=..., device=...)` decodes it.

Points that are easy to miss:

- A packed `X` gives packed sequences back from every function in `ersatz`, and packed references from `deep_lift_shap` and `pisa` with `return_references=True`. Pass `return_packed=False` to get an int8 one-hot encoding instead.
- A `func` or `shuffle_fn` you write receives the int8 one-hot encoding, not the `PackedSequence`.
- Decoding costs time on every batch. Packing pays off where many sequences are held at once, as with genome-wide windows or precomputed references, not where a few are.

| Function | What it does | The argument that matters |
|---|---|---|
| `PackedSequence.from_one_hot` | packs a one-hot encoding | `alphabet` |
| `PackedSequence.to_one_hot` | decodes, in a dtype and on a device | `dtype`, `device` |
| `extract_loci` | reads windows of a FASTA file | `return_packed` |
| `dinucleotide_shuffle` | draws shuffles, packed as they are drawn | `return_packed`, `n_jobs` |
| `PackedSequence.save` / `load` | writes and reads a directory of `.npy` files | `mmap` |

If you remember one thing: pass a `PackedSequence` wherever you would pass `X`, and the results do not change.

The docstring of `tangermeme.utils.PackedSequence` describes the byte layout, and Tutorial A3 covers the references of `deep_lift_shap`.